In [0]:
dbutils.widgets.text("catalog", "claims")
catalog = dbutils.widgets.get("catalog")

spark.sql(f"USE CATALOG {catalog}")

In [0]:
from pyspark.sql import functions as F

checkpoint_path = f"/Volumes/{catalog}/raw/landing/claims/checkpoints/"
source_path = f"/Volumes/{catalog}/raw/landing/claims_raw/"
schema_path = f"/Volumes/{catalog}/raw/landing/claims/schema"
df = spark.readStream.format('cloudFiles') \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaLocation", schema_path) \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("cloudFiles.rescuedDataColumn", "_rescued_data") \
    .load(source_path)


df = df.withColumns({
    "_source_file" : F.col("_metadata.file_path"),
    "_ingest_ts" : F.current_timestamp()
})

df.writeStream.format('delta') \
    .option('checkpointLocation', checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(f'{catalog}.bronze.claims')